# Repository guide: Fadhma quarter-SPK009

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution. The notebook records the completed run; supplied model metadata stops at step 126 and does not include selected checkpoint-252.


# Fadhma-300M → Tarifit V1.2 — Quarter-Bible Transfer Experiment

This experiment tests a stronger reduction of the dominant Bible speaker/domain.

- Starting checkpoint: `agbalu/Fadhma-300M`
- Keep 100% of non-Bible training data
- Keep approximately 25% of SPK009 Bible duration
- Preserve every SPK009 recording
- Same frozen 129-segment validation set
- Fresh 34-class Tarifit CTC head
- No augmentation
- Feature encoder frozen
- LR 3e-5
- Effective batch size 16
- Up to 8 epochs
- Early stopping patience 2
- Best checkpoint selected by CER

Because this is a 25%-Bible condition, it is a stronger data-scarcity ablation rather than a perfectly matched 50%-Bible cross-model comparison.


In [ ]:
# Cell 1 — Install dependencies
!pip -q install "transformers==4.57.1" "datasets==4.4.1" "accelerate>=1.10,<2" "jiwer==4.0.0" "safetensors>=0.4.5" "soundfile>=0.12.1"
print("✓ Dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
✓ Dependencies installed.


In [ ]:
# Cell 2 — Mount Drive and define paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm")
METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2.csv"
FROZEN_METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2_train_val_frozen.csv"
TOKENIZER_DIR = PROJECT_ROOT / "data" / "processed" / "mms_tokenizer_v1_2"
DATASET_CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_2"
CACHE_MANIFEST_PATH = DATASET_CACHE_DIR / "cache_manifest.json"

OUTPUT_DIR = PROJECT_ROOT / "models" / "fadhma_300m_tarifit_v1_2_quarter_bible"
RESULTS_DIR = PROJECT_ROOT / "results" / "fadhma_300m_tarifit_v1_2_quarter_bible"
SUBSET_METADATA_PATH = RESULTS_DIR / "quarter_bible_training_subset.csv"
SUBSET_IDS_PATH = RESULTS_DIR / "quarter_bible_training_segment_ids.txt"

FULL_FADHMA_SUMMARY_PATH = PROJECT_ROOT / "results" / "fadhma_300m_tarifit_v1_2_transfer" / "experiment_summary.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_ID = "agbalu/Fadhma-300M"
SOURCE_BASE_MODEL = "ylacombe/omniASR_W2V_300M_SSL"

print("Base model:", BASE_MODEL_ID)
print("Output:", OUTPUT_DIR)


Mounted at /content/drive
Base model: agbalu/Fadhma-300M
Output: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_2_quarter_bible


In [ ]:
# Cell 3 — Verify versions, GPU, and seed
import sys
import json
import random
import hashlib
import numpy as np
import pandas as pd
import torch
import transformers
import datasets

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())

assert transformers.__version__ == "4.57.1"
assert datasets.__version__ == "4.4.1"

if not torch.cuda.is_available():
    raise RuntimeError("Switch Colab to a GPU runtime.")

print("GPU:", torch.cuda.get_device_name(0))


Transformers: 4.57.1
Datasets: 4.4.1
CUDA: True
GPU: Tesla T4


In [ ]:
# Cell 4 — Load the frozen V1.2 split
frozen_df = pd.read_csv(FROZEN_METADATA_PATH)

for col in ["segment_id", "recording_id", "speaker_group_id", "dataset_split", "transcription"]:
    frozen_df[col] = frozen_df[col].fillna("").astype(str).str.strip()

frozen_df["dataset_split"] = frozen_df["dataset_split"].str.lower()
frozen_df["duration_seconds"] = pd.to_numeric(frozen_df["duration_seconds"], errors="raise")

full_train_df = frozen_df[frozen_df["dataset_split"].eq("train")].copy()
val_df = frozen_df[frozen_df["dataset_split"].eq("validation")].copy()

assert len(full_train_df) == 1754
assert len(val_df) == 129

print("Full train:", len(full_train_df), "segments /", round(full_train_df["duration_seconds"].sum()/3600, 3), "h")
print("Validation:", len(val_df), "segments /", round(val_df["duration_seconds"].sum()/3600, 3), "h")


Full train: 1754 segments / 5.223 h
Validation: 129 segments / 0.298 h


In [ ]:
# Cell 5 — Audit full training data
speaker_stats = (
    full_train_df
    .groupby("speaker_group_id")
    .agg(
        segments=("segment_id", "count"),
        seconds=("duration_seconds", "sum"),
        recordings=("recording_id", "nunique"),
    )
    .reset_index()
)

speaker_stats["hours"] = speaker_stats["seconds"] / 3600
speaker_stats["duration_share_pct"] = 100 * speaker_stats["seconds"] / speaker_stats["seconds"].sum()
display(speaker_stats)

bible_full_df = full_train_df[full_train_df["speaker_group_id"].eq("SPK009")].copy()
other_train_df = full_train_df[~full_train_df["speaker_group_id"].eq("SPK009")].copy()

print("Bible hours:", round(bible_full_df["duration_seconds"].sum()/3600, 3))
print("Non-Bible hours:", round(other_train_df["duration_seconds"].sum()/3600, 3))


,speaker_group_id,segments,seconds,recordings,hours,duration_share_pct
0,SPK001,96,853.656,4,0.237127,4.540279
1,SPK002,186,1251.376,4,0.347604,6.655604
2,SPK009,1472,16696.808,44,4.638002,88.804117


Bible hours: 4.638
Non-Bible hours: 0.585


In [ ]:
# Cell 6 — Select approximately 25% of SPK009 duration within every recording
BIBLE_KEEP_FRACTION = 0.25

def select_fraction_per_recording(frame, fraction=0.25, seed=42):
    kept_parts = []
    audit_rows = []

    for recording_id, group in frame.groupby("recording_id", sort=True):
        stable_int = int(
            hashlib.sha256(f"{seed}:{recording_id}".encode("utf-8")).hexdigest()[:8],
            16,
        )

        rng = np.random.default_rng(stable_int)
        shuffled = group.iloc[rng.permutation(len(group))].copy()

        total_seconds = float(group["duration_seconds"].sum())
        target_seconds = fraction * total_seconds

        cumulative = shuffled["duration_seconds"].cumsum().to_numpy()
        cutoff = int(np.searchsorted(cumulative, target_seconds, side="left"))
        cutoff = min(cutoff, len(shuffled) - 1)

        kept = shuffled.iloc[: cutoff + 1].copy()
        kept_parts.append(kept)

        audit_rows.append({
            "recording_id": recording_id,
            "full_segments": len(group),
            "kept_segments": len(kept),
            "full_seconds": total_seconds,
            "target_seconds": target_seconds,
            "kept_seconds": float(kept["duration_seconds"].sum()),
            "kept_fraction_duration": float(kept["duration_seconds"].sum()) / total_seconds,
        })

    return pd.concat(kept_parts, ignore_index=True), pd.DataFrame(audit_rows)

bible_quarter_df, bible_sampling_audit = select_fraction_per_recording(
    bible_full_df,
    fraction=BIBLE_KEEP_FRACTION,
    seed=SEED,
)

display(bible_sampling_audit)

print("Bible full hours:", round(bible_full_df["duration_seconds"].sum()/3600, 3))
print("Bible retained hours:", round(bible_quarter_df["duration_seconds"].sum()/3600, 3))
print(
    "Actual retained Bible duration:",
    f"{100*bible_quarter_df['duration_seconds'].sum()/bible_full_df['duration_seconds'].sum():.2f}%"
)

assert set(bible_quarter_df["recording_id"]) == set(bible_full_df["recording_id"])
assert bible_quarter_df["segment_id"].is_unique


,recording_id,full_segments,kept_segments,full_seconds,target_seconds,kept_seconds,kept_fraction_duration
0,REC094,19,4,221.328,55.33200,56.580,0.255639
1,REC095,25,7,274.784,68.69600,72.760,0.264790
2,REC096,16,4,187.030,46.75750,47.312,0.252965
3,REC097,22,6,258.385,64.59625,74.840,0.289645
4,REC098,43,12,486.318,121.57950,128.660,0.264559
5,REC099,34,9,362.224,90.55600,90.652,0.250265
6,REC100,23,5,267.850,66.96250,68.956,0.257443
7,REC101,29,7,327.072,81.76800,87.528,0.267611
8,REC102,29,8,334.294,83.57350,97.004,0.290176
9,REC103,33,9,407.965,101.99125,116.104,0.284593


Bible full hours: 4.638
Bible retained hours: 1.242
Actual retained Bible duration: 26.78%


In [ ]:
# Cell 7 — Build and freeze the quarter-Bible subset
quarter_train_df = pd.concat(
    [other_train_df, bible_quarter_df],
    ignore_index=True,
)

quarter_train_df = (
    quarter_train_df
    .sort_values(["speaker_group_id", "recording_id", "segment_id"])
    .reset_index(drop=True)
)

assert quarter_train_df["segment_id"].is_unique
assert set(other_train_df["segment_id"]) <= set(quarter_train_df["segment_id"])
assert not (set(quarter_train_df["segment_id"]) & set(val_df["segment_id"]))

quarter_train_df.to_csv(
    SUBSET_METADATA_PATH,
    index=False,
    encoding="utf-8",
)

with open(SUBSET_IDS_PATH, "w", encoding="utf-8") as f:
    for segment_id in quarter_train_df["segment_id"]:
        f.write(str(segment_id) + "\n")

subset_hash = hashlib.sha256(
    "\n".join(
        quarter_train_df["segment_id"].astype(str).tolist()
    ).encode("utf-8")
).hexdigest()

print("Quarter-Bible train segments:", len(quarter_train_df))
print("Quarter-Bible train hours:", round(quarter_train_df["duration_seconds"].sum()/3600, 3))
print("Bible retained hours:", round(bible_quarter_df["duration_seconds"].sum()/3600, 3))
print("Non-Bible hours:", round(other_train_df["duration_seconds"].sum()/3600, 3))
print("Subset SHA256:", subset_hash)
print("✓ Quarter-Bible subset frozen.")


Quarter-Bible train segments: 667
Quarter-Bible train hours: 1.827
Bible retained hours: 1.242
Non-Bible hours: 0.585
Subset SHA256: 383eff818f2fc6fec8a57d63a754bfcdf06e6e3ce18bfd456442801fe3ec8d92
✓ Quarter-Bible subset frozen.


In [ ]:
# Cell 8 — Verify speaker independence and test isolation
train_speakers = set(quarter_train_df["speaker_group_id"].dropna())
val_speakers = set(val_df["speaker_group_id"].dropna())

assert not (train_speakers & val_speakers)

master_df = pd.read_csv(METADATA_PATH)

for col in ["dataset_split", "speaker_group_id"]:
    master_df[col] = master_df[col].fillna("").astype(str).str.strip()

test_speakers = set(
    master_df.loc[
        master_df["dataset_split"].str.lower().eq("test"),
        "speaker_group_id",
    ]
)

print("Train/validation overlap:", bool(train_speakers & val_speakers))
print("Train/test overlap:", bool(train_speakers & test_speakers))
print("Validation/test overlap:", bool(val_speakers & test_speakers))

assert not (train_speakers & test_speakers)
assert not (val_speakers & test_speakers)


Train/validation overlap: False
Train/test overlap: False
Validation/test overlap: False


In [ ]:
# Cell 9 — Load the exact existing V1.2 tokenizer
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

FINAL_LETTERS = [
    "a","b","c","d","ḍ","e","ɛ","f","g","h","ḥ","i","j","k","l","m",
    "n","p","q","r","s","t","ṭ","u","v","w","x","y","z","ɣ","ʷ"
]

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    existing_vocab = json.load(f)

expected_tokens = set(FINAL_LETTERS) | {"|", "[UNK]", "[PAD]"}

assert len(existing_vocab) == 34
assert set(existing_vocab.keys()) == expected_tokens
assert sorted(existing_vocab.values()) == list(range(34))

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("Tokenizer size:", len(tokenizer))
print("PAD id:", tokenizer.pad_token_id)


Tokenizer size: 34
PAD id: 33


In [ ]:
# Cell 10 — Load cached V1.2 waveforms and labels
from datasets import load_from_disk, DatasetDict

metadata_sha256 = hashlib.sha256(FROZEN_METADATA_PATH.read_bytes()).hexdigest()

with open(CACHE_MANIFEST_PATH, "r", encoding="utf-8") as f:
    cache_manifest = json.load(f)

assert cache_manifest.get("metadata_sha256") == metadata_sha256

full_dataset = load_from_disk(str(DATASET_CACHE_DIR))

assert len(full_dataset["train"]) == 1754
assert len(full_dataset["validation"]) == 129

print("Cached train:", len(full_dataset["train"]))
print("Cached validation:", len(full_dataset["validation"]))


Cached train: 1754
Cached validation: 129


In [ ]:
# Cell 11 — Map quarter-Bible IDs onto the cached training set
cached_train_ids = list(full_dataset["train"]["segment_id"])
id_to_index = {segment_id: i for i, segment_id in enumerate(cached_train_ids)}

missing_from_cache = [
    segment_id
    for segment_id in quarter_train_df["segment_id"]
    if segment_id not in id_to_index
]

print("Selected IDs missing from cache:", len(missing_from_cache))
assert not missing_from_cache

quarter_train_indices = [
    id_to_index[segment_id]
    for segment_id in quarter_train_df["segment_id"]
]

quarter_train_ds = full_dataset["train"].select(quarter_train_indices)
val_ds = full_dataset["validation"]

assert list(quarter_train_ds["segment_id"]) == list(quarter_train_df["segment_id"])

experiment_dataset = DatasetDict({
    "train": quarter_train_ds,
    "validation": val_ds,
})

print(experiment_dataset)
print(
    "Cached train hours:",
    round(
        sum(experiment_dataset["train"]["input_length"])
        / 16000
        / 3600,
        3,
    )
)


Selected IDs missing from cache: 0
DatasetDict({
    train: Dataset({
        features: ['segment_id', 'input_values', 'input_length', 'labels'],
        num_rows: 667
    })
    validation: Dataset({
        features: ['segment_id', 'input_values', 'input_length', 'labels'],
        num_rows: 129
    })
})
Cached train hours: 1.827


In [ ]:
# Cell 12 — Load Fadhma with a fresh Tarifit CTC head
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,
    apply_spec_augment=False,
)

model.freeze_feature_encoder()
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100*trainable_params/total_params:.4f}%")
print("CTC vocab:", model.config.vocab_size)
print("SpecAugment:", model.config.apply_spec_augment)

assert model.config.vocab_size == 34
assert model.config.apply_spec_augment is False
assert 300_000_000 < total_params < 330_000_000
assert trainable_params > 300_000_000


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at agbalu/Fadhma-300M and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([40]) in the checkpoint and torch.Size([34]) in the model instantiated
- lm_head.weight: found shape torch.Size([40, 1024]) in the checkpoint and torch.Size([34, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 315,473,570
Trainable parameters: 311,263,394
Trainable percentage: 98.6654%
CTC vocab: 34
SpecAugment: False


In [ ]:
# Cell 13 — Check CTC feasibility
def minimum_ctc_frames(labels):
    labels = list(labels)
    repeats = sum(
        labels[i] == labels[i-1]
        for i in range(1, len(labels))
    )
    return len(labels) + repeats

def model_output_frames(input_samples):
    return int(
        model._get_feat_extract_output_lengths(
            torch.tensor(int(input_samples))
        ).item()
    )

def find_bad(split_ds):
    bad = []
    for i, ex in enumerate(split_ds):
        out_frames = model_output_frames(ex["input_length"])
        min_frames = minimum_ctc_frames(ex["labels"])

        if out_frames < min_frames:
            bad.append({
                "index": i,
                "segment_id": ex["segment_id"],
                "output_frames": out_frames,
                "minimum_ctc_frames": min_frames,
            })
    return bad

bad_train = find_bad(experiment_dataset["train"])
bad_val = find_bad(experiment_dataset["validation"])

print("CTC-infeasible train:", len(bad_train))
print("CTC-infeasible validation:", len(bad_val))

assert not bad_train
assert not bad_val


CTC-infeasible train: 0
CTC-infeasible validation: 0


In [ ]:
# Cell 14 — Define dynamic CTC padding
from dataclasses import dataclass
from typing import Any, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [
            {"input_values": f["input_values"]}
            for f in features
        ]

        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        batch["labels"] = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)
print("✓ Collator ready.")


✓ Collator ready.


In [ ]:
# Cell 15 — Define WER and CER
from jiwer import wer, cer

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = [
        x.strip()
        for x in processor.batch_decode(pred_ids)
    ]

    ref_str = [
        x.strip()
        for x in processor.batch_decode(
            label_ids,
            group_tokens=False,
        )
    ]

    return {
        "wer": wer(ref_str, pred_str),
        "cer": cer(ref_str, pred_str),
    }

print("✓ Metrics ready.")


✓ Metrics ready.


In [ ]:
# Cell 16 — Configure matched Fadhma training
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=8,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001,
)

print("Train examples:", len(experiment_dataset["train"]))
print("Validation examples:", len(experiment_dataset["validation"]))
print("Learning rate:", training_args.learning_rate)
print("Max epochs:", training_args.num_train_epochs)
print("Effective batch size:", 16)


Train examples: 667
Validation examples: 129
Learning rate: 3e-05
Max epochs: 8
Effective batch size: 16


In [ ]:
# Cell 17 — Create Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=experiment_dataset["train"],
    eval_dataset=experiment_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[early_stopping],
)

print("✓ Trainer ready.")


✓ Trainer ready.


In [ ]:
# Cell 18 — Start or resume training
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = (
    get_last_checkpoint(str(OUTPUT_DIR))
    if OUTPUT_DIR.exists()
    else None
)

if last_checkpoint:
    print("Resuming from:", last_checkpoint)
    train_result = trainer.train(
        resume_from_checkpoint=last_checkpoint
    )
else:
    print("Starting fresh training.")
    train_result = trainer.train()

print("Training finished.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation CER:", trainer.state.best_metric)


Starting fresh training.


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Epoch,Training Loss,Validation Loss,Wer,Cer
1,11.123200,3.395567,1.000000,1.000000
2,2.999300,3.366959,1.000000,1.000000
3,1.483700,3.301336,0.982143,0.501728


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Epoch,Training Loss,Validation Loss,Wer,Cer
1,11.123200,3.395567,1.000000,1.000000
2,2.999300,3.366959,1.000000,1.000000
3,1.483700,3.301336,0.982143,0.501728
4,0.884200,3.490950,0.946429,0.487258
5,0.636600,3.704955,0.944728,0.485168
6,0.535600,3.603988,0.926020,0.476726
7,0.507900,3.909547,0.945153,0.486293
8,0.463600,3.754704,0.929847,0.481068


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Training finished.
Best checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_2_quarter_bible/checkpoint-252
Best validation CER: 0.47672642495377443


In [ ]:
# Cell 19 — Show and save epoch results
rows = []
last_train_loss = None

for log in trainer.state.log_history:
    if "loss" in log and "eval_loss" not in log:
        last_train_loss = log["loss"]

    if "eval_loss" in log:
        rows.append({
            "epoch": log.get("epoch"),
            "training_loss": last_train_loss,
            "validation_loss": log.get("eval_loss"),
            "WER": log.get("eval_wer"),
            "CER": log.get("eval_cer"),
        })

history_df = pd.DataFrame(rows)
display(history_df)

history_df.to_csv(
    RESULTS_DIR / "training_history.csv",
    index=False,
)


,epoch,training_loss,validation_loss,WER,CER
0,1.0,11.1232,3.395567,1.000000,1.000000
1,2.0,2.9993,3.366959,1.000000,1.000000
2,3.0,1.4837,3.301336,0.982143,0.501728
3,4.0,0.8842,3.490950,0.946429,0.487258
4,5.0,0.6366,3.704955,0.944728,0.485168
5,6.0,0.5356,3.603988,0.926020,0.476726
6,7.0,0.5079,3.909547,0.945153,0.486293
7,8.0,0.4636,3.754704,0.929847,0.481068


In [ ]:
# Cell 20 — Evaluate and save best checkpoint
best_metrics = trainer.evaluate(
    eval_dataset=experiment_dataset["validation"]
)

best_wer = float(best_metrics["eval_wer"])
best_cer = float(best_metrics["eval_cer"])

print(f"Best validation WER: {best_wer:.6f} ({best_wer*100:.2f}%)")
print(f"Best validation CER: {best_cer:.6f} ({best_cer*100:.2f}%)")
print("Selected checkpoint:", trainer.state.best_model_checkpoint)

BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

trainer.save_model(str(BEST_MODEL_DIR))
processor.save_pretrained(str(BEST_MODEL_DIR))

print("Saved:", BEST_MODEL_DIR)


Best validation WER: 0.926020 (92.60%)
Best validation CER: 0.476726 (47.67%)
Selected checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_2_quarter_bible/checkpoint-252
Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_2_quarter_bible/best_model


In [ ]:
# Cell 21 — Save predictions and compute space-insensitive CER
prediction_output = trainer.predict(
    experiment_dataset["validation"]
)

pred_ids = np.argmax(
    prediction_output.predictions,
    axis=-1,
)

label_ids = prediction_output.label_ids.copy()
label_ids[label_ids == -100] = tokenizer.pad_token_id

predictions = [
    x.strip()
    for x in processor.batch_decode(pred_ids)
]

references = [
    x.strip()
    for x in processor.batch_decode(
        label_ids,
        group_tokens=False,
    )
]

prediction_df = pd.DataFrame({
    "segment_id": experiment_dataset["validation"]["segment_id"],
    "reference": references,
    "prediction": predictions,
})

prediction_df["segment_wer"] = [
    wer(r, p)
    for r, p in zip(references, predictions)
]

prediction_df["segment_cer"] = [
    cer(r, p)
    for r, p in zip(references, predictions)
]

prediction_df["segment_cer_no_spaces"] = [
    cer(
        r.replace(" ", ""),
        p.replace(" ", ""),
    )
    for r, p in zip(references, predictions)
]

global_cer_no_spaces = cer(
    [r.replace(" ", "") for r in references],
    [p.replace(" ", "") for p in predictions],
)

prediction_df.to_csv(
    RESULTS_DIR / "validation_predictions.csv",
    index=False,
    encoding="utf-8",
)

display(prediction_df.head(20))

print("Standard CER:", f"{best_cer*100:.2f}%")
print("Space-insensitive CER:", f"{global_cer_no_spaces*100:.2f}%")


,segment_id,reference,prediction,segment_wer,segment_cer,segment_cer_no_spaces
0,REC090_SEG0010,ssalamuɛlikum necc meryem,ssaram u ɛlikum necc meryamr,1.333333,0.200000,0.130435
1,REC090_SEG0011,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,aqa ruxxa knayen uɛecrin sanadi hulanda,0.571429,0.125000,0.117647
2,REC090_SEG0012,mercex ak nmis n jjiran usiɣ ḍ zi lmeɣrib umi ...,mercex ak misen jjiran usiɣ d zi lmeɣrib umi i...,0.640000,0.278261,0.274725
3,REC090_SEG0013,umi wsiɣd dda ufix manayenni wa dji ca min ira...,umi wsiɣ dda ufix ufix mana yenniwadjica mir i...,0.642857,0.250000,0.215686
4,REC090_SEG0014,a necc mammec ira djjix ḍi lmeɣrib wadji manay...,necc amamc ira djix tlmeɣrib wadji mana yenn u...,0.818182,0.250000,0.220000
5,REC090_SEG0015,necc ḍi lmeɣrib ira ɣari lḥurriya inu ira ɣari...,unec ḍi lmeɣrib ira ɣari lḥurriya inu ila ɣar ...,0.703704,0.184211,0.198413
6,REC090_SEG0016,ḍi lmeɣrib neccin mammec ira niɛicc,ḍi lmaɣrim nccin mamciraniɛicwm,0.833333,0.257143,0.233333
7,REC090_SEG0017,ak baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥway...,ag baba d yemma waɣanexca ɣan x ca neḥwa ijj n...,0.750000,0.217647,0.192593
8,REC090_SEG0018,lmuhim wsiɣd,muhimwsiɣ d,1.000000,0.250000,0.090909
9,REC090_SEG0019,necc ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ḍi ṭi...,n ccira ɛemma swawsiɣ d ɣa urubbaw siɣ dditiya...,0.821429,0.246575,0.142857


Standard CER: 47.67%
Space-insensitive CER: 46.08%


In [ ]:
# Cell 22 — Compare quarter-Bible Fadhma with full-data Fadhma
comparison_rows = []

full_fadhma_wer = None
full_fadhma_cer = None

if FULL_FADHMA_SUMMARY_PATH.exists():
    with open(
        FULL_FADHMA_SUMMARY_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        full_summary = json.load(f)

    full_fadhma_wer = float(
        full_summary["best_validation_wer"]
    )

    full_fadhma_cer = float(
        full_summary["best_validation_cer"]
    )

    comparison_rows.append({
        "experiment": "Fadhma full V1.2",
        "WER": full_fadhma_wer,
        "CER": full_fadhma_cer,
    })

comparison_rows.append({
    "experiment": "Fadhma quarter-Bible V1.2",
    "WER": best_wer,
    "CER": best_cer,
})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["WER_percent"] = 100 * comparison_df["WER"]
comparison_df["CER_percent"] = 100 * comparison_df["CER"]

display(comparison_df)

if full_fadhma_wer is not None:
    print(
        "WER delta:",
        f"{100*(best_wer-full_fadhma_wer):+.2f} percentage points",
    )
    print(
        "CER delta:",
        f"{100*(best_cer-full_fadhma_cer):+.2f} percentage points",
    )

comparison_df.to_csv(
    RESULTS_DIR / "validation_comparison.csv",
    index=False,
)


,experiment,WER,CER,WER_percent,CER_percent
0,Fadhma full V1.2,0.896259,0.462497,89.625850,46.249699
1,Fadhma quarter-Bible V1.2,0.926020,0.476726,92.602041,47.672642


WER delta: +2.98 percentage points
CER delta: +1.42 percentage points


In [ ]:
# Cell 23 — Save experiment summary
summary = {
    "experiment": "Fadhma-300M Kabyle to Tarifit V1.2 quarter-Bible ablation",
    "transfer_checkpoint": BASE_MODEL_ID,
    "source_language": "Kabyle",
    "target_language": "Tarifit",
    "original_ssl_backbone": SOURCE_BASE_MODEL,
    "seed": SEED,
    "full_train_segments": int(len(full_train_df)),
    "full_train_hours": float(full_train_df["duration_seconds"].sum()/3600),
    "quarter_bible_train_segments": int(len(quarter_train_df)),
    "quarter_bible_train_hours": float(quarter_train_df["duration_seconds"].sum()/3600),
    "full_bible_hours": float(bible_full_df["duration_seconds"].sum()/3600),
    "retained_bible_hours": float(bible_quarter_df["duration_seconds"].sum()/3600),
    "retained_bible_duration_fraction": float(
        bible_quarter_df["duration_seconds"].sum()
        / bible_full_df["duration_seconds"].sum()
    ),
    "non_bible_train_hours": float(other_train_df["duration_seconds"].sum()/3600),
    "validation_segments": int(len(val_df)),
    "training_subset_sha256": subset_hash,
    "augmentation": "none",
    "learning_rate": 3e-5,
    "max_epochs": 8,
    "effective_batch_size": 16,
    "early_stopping_patience": 2,
    "checkpoint_selection_metric": "CER",
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_wer": best_wer,
    "best_validation_cer": best_cer,
    "space_insensitive_validation_cer": float(global_cer_no_spaces),
}

if full_fadhma_wer is not None:
    summary["full_data_reference_wer"] = full_fadhma_wer
    summary["full_data_reference_cer"] = full_fadhma_cer
    summary["delta_wer_vs_full"] = float(best_wer - full_fadhma_wer)
    summary["delta_cer_vs_full"] = float(best_cer - full_fadhma_cer)

with open(
    RESULTS_DIR / "experiment_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(summary, indent=2))


{
  "experiment": "Fadhma-300M Kabyle to Tarifit V1.2 quarter-Bible ablation",
  "transfer_checkpoint": "agbalu/Fadhma-300M",
  "source_language": "Kabyle",
  "target_language": "Tarifit",
  "original_ssl_backbone": "ylacombe/omniASR_W2V_300M_SSL",
  "seed": 42,
  "full_train_segments": 1754,
  "full_train_hours": 5.222733333333333,
  "quarter_bible_train_segments": 667,
  "quarter_bible_train_hours": 1.8268630555555556,
  "full_bible_hours": 4.638002222222221,
  "retained_bible_hours": 1.2421319444444445,
  "retained_bible_duration_fraction": 0.2678161598312684,
  "non_bible_train_hours": 0.5847311111111112,
  "validation_segments": 129,
  "training_subset_sha256": "383eff818f2fc6fec8a57d63a754bfcdf06e6e3ce18bfd456442801fe3ec8d92",
  "augmentation": "none",
  "learning_rate": 3e-05,
  "max_epochs": 8,
  "effective_batch_size": 16,
  "early_stopping_patience": 2,
  "checkpoint_selection_metric": "CER",
  "best_checkpoint": "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/m

## Interpretation

Compare this primarily with the **full-data Fadhma V1.2** result.

Because this retains only about 25% of the Bible component, it is a stronger data-scarcity condition than the 50%-Bible MMS/Omni experiments.

If performance remains similar or improves, that would suggest the Kabyle-adapted Fadhma initialization can work with substantially less dominant Tarifit training speech. If performance degrades, the removed target-language hours remain important despite related-language initialization.


Reducing the dominant read-speech component did not have a uniform effect across adaptation strategies. A moderate 50% reduction slightly improved MMS adapter fine-tuning, whereas reducing the available target-language speech degraded both OmniASR and Fadhma full-encoder fine-tuning. In the Fadhma experiment, retaining approximately one quarter of the Bible component increased WER from 89.63% to 92.60% and CER from 46.25% to 47.67%. These results suggest a trade-off between corpus balance and the amount of target-language supervision available to the model.